In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os, time, json
from datetime import datetime, timezone

spark = (SparkSession.builder
         .appName("hafta6")
         .master("local[4]")
         .getOrCreate())

PARTI = "/home/jovyan/work/yeni_parti.parquet"

parti = spark.read.parquet(PARTI)

parti.createOrReplaceTempView("parti")

print("Parti yeri  :", PARTI)
print("Satir sayisi:", parti.count())

parti.printSchema()

Parti yeri  : /home/jovyan/work/yeni_parti.parquet
Satir sayisi: 205000
root
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- text: string (nullable = true)



In [9]:
parti.show(5, truncate=30)
 
(parti.groupBy("verified_purchase")
      .count()
      .orderBy(F.desc("count"))
      .show())


+------+-------------+-----------+----------------------------+------------+-----------------+------------------------------+
|rating|    timestamp|parent_asin|                     user_id|helpful_vote|verified_purchase|                          text|
+------+-------------+-----------+----------------------------+------------+-----------------+------------------------------+
|   5.0|1655321251964| B075X8TPMN|AFMICOEPLTALM7JDYKGJCDKYEOSQ|           0|             True|The title says it all. This...|
|   5.0|1620845245724| B07NK6HYN3|AG5S22MEHCFR43B5VZ2PCR2QNBLA|           2|             True|Love this cover. It’s the p...|
|   3.0|1657851175157| B082YJ71VK|AH5I3FAB2XCJJYJUYLCXAP34ENYA|           1|             true|I like this basket, it hold...|
|   5.0|1624931950868| B0C5B1WSGJ|AGKYR37SFCZPY7D3IGYNTGO4DEBQ|           0|             true|Great for plus size moms an...|
|   5.0|1657549446676| B073X65L2M|AHO4BO7ECIZ37HGAYYX7OHO2J2FQ|           0|                Y|Absolutely love these po

In [10]:
MS_MIN = 830908800000        # 1 Mayis 1996
MS_MAX = 1696118400000       # 1 Ekim 2023
SANIYE_ESIGI = 100000000000  # bunun altindaki zaman damgasi saniye sayilir
 
KONTROLLER = {
    "zaman_eksik":          "`timestamp` IS NULL",
    "metin_bos":            "text IS NULL OR TRIM(text) = ''",
    "puan_gecersiz":        "rating IS NULL OR rating < 1 OR rating > 5",
    "oy_negatif":           "helpful_vote < 0",
    "zaman_aralik_disi":    f"`timestamp` < {MS_MIN} OR `timestamp` > {MS_MAX}",
    "zaman_saniye_gibi":    f"`timestamp` > 0 AND `timestamp` < {SANIYE_ESIGI}",
    "zaman_gelecek":        f"`timestamp` > {MS_MAX}",
    "verified_bicim_bozuk": ("verified_purchase IS NULL OR "
                             "CAST(verified_purchase AS STRING) NOT IN ('true', 'false')"),
}
 
def kontrol(df, goster=True):
    df.createOrReplaceTempView("kontrol_tablosu")
    parcalar = [f"SUM(CASE WHEN {kosul} THEN 1 ELSE 0 END) AS {ad}"
                for ad, kosul in KONTROLLER.items()]
    sorgu = "SELECT COUNT(*) AS toplam, " + ", ".join(parcalar) + " FROM kontrol_tablosu"
    sonuc = spark.sql(sorgu).first().asDict()
    sonuc = {ad: int(deger or 0) for ad, deger in sonuc.items()}
    if goster:
        for ad, deger in sonuc.items():
            print(f"{ad:24} {deger:>10,}")
    return sonuc
 
t0 = time.time()
rapor_once = kontrol(parti)
print(f"Sure: {time.time() - t0:.1f} sn")


toplam                      205,000
zaman_eksik                   3,000
metin_bos                     2,500
puan_gecersiz                 2,000
oy_negatif                    1,000
zaman_aralik_disi             2,500
zaman_saniye_gibi             2,000
zaman_gelecek                   500
verified_bicim_bozuk        153,910
Sure: 2.0 sn


In [11]:
def yinelen_say(df):
    toplam = df.count()
    tam = toplam - df.dropDuplicates().count()
    anahtar = toplam - df.dropDuplicates(["user_id", "parent_asin", "timestamp"]).count()
    return tam, anahtar
 
tam, anahtar = yinelen_say(parti)
print("Birebir ayni satir fazlasi :", tam)
print("Anahtara gore fazla        :", anahtar)
print("Sadece anahtar ayni olan   :", anahtar - tam)
 
cakisan = (parti.groupBy("user_id", "parent_asin", "timestamp")
           .agg(F.countDistinct("rating").alias("farkli_puan"))
           .filter("farkli_puan > 1"))
print("Ayni yorum iki farkli puanla gelen:", cakisan.count())


Birebir ayni satir fazlasi : 4000
Anahtara gore fazla        : 5000
Sadece anahtar ayni olan   : 1000
Ayni yorum iki farkli puanla gelen: 1000


In [12]:
rapor_once["yinelenen_tam_fazla"] = tam
rapor_once["yinelenen_anahtar_fazla"] = anahtar
 
with open("kural_raporu_once.json", "w") as f:
    json.dump(rapor_once, f, indent=2)
print("Kaydedildi:", rapor_once)


Kaydedildi: {'toplam': 205000, 'zaman_eksik': 3000, 'metin_bos': 2500, 'puan_gecersiz': 2000, 'oy_negatif': 1000, 'zaman_aralik_disi': 2500, 'zaman_saniye_gibi': 2000, 'zaman_gelecek': 500, 'verified_bicim_bozuk': 153910, 'yinelenen_tam_fazla': 4000, 'yinelenen_anahtar_fazla': 5000}
